# SCAR / M3 trên Google Colab
Notebook dùng cùng cấu hình và CLI với repository. Chọn runtime A100 nếu có; mặc định batch 16, LR 0.001, ảnh 128². Xem README để hiểu hợp đồng nhãn và giới hạn HD95. Trước khi dùng bản trên GitHub, bảo đảm repository của bạn đã chứa phiên bản SCAR này.

In [ ]:
from pathlib import Path
import subprocess, sys, os
REPO = Path("/content/SCAR")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/thanhquan123hi1/SCAR.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Dữ liệu
Có thể mount Drive hoặc dùng ổ local của runtime. Đặt DATA_ROOT trỏ tới cache gồm bSSFP/LGE/T2w; mỗi nhánh có train_npz và test_vol_h5. Cache legacy dùng lớp 2=scar, 3=edema; cache SCAR mới có metadata canonical. Không đặt dataset lớn trong thư mục repository.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/MyoPS380/Processed_data"
RUN_ROOT = "/content/drive/MyDrive/SCAR_runs"
RUN_ID = "m3_a100_01"
LABEL_ORDER = "legacy"  # "auto" for metadata-tagged canonical SCAR cache
assert Path(DATA_ROOT).is_dir(), DATA_ROOT

## Kiểm tra nhẹ trước khi train
Đây là kiểm tra cấu trúc nhỏ, không phải mô hình production hoặc đo chất lượng segmentation.

In [ ]:
subprocess.run([sys.executable, "tools/sanity_check.py", "--profile", "testing", "--device", "cuda" if torch.cuda.is_available() else "cpu", "--amp", "auto", "--image-size", "32"], check=True)

## Train production và đánh giá volume
Giữ batch 16, 300 epoch, LR 0.001 theo base.yaml. Có thể thêm --skip-evaluate trong giai đoạn chọn cấu hình và chỉ đánh giá test ở cuối. Missing geometry cho HD95 mm=null; không tự coi voxel là mm.

In [ ]:
subprocess.run([sys.executable, "run_all.py", "--config", "training/config/models/cmspa_net.yaml", "--run-id", RUN_ID, "--run-root", RUN_ROOT, "--data-root", DATA_ROOT, "--list-dir", "data/processed/splits", "--skip-cache", "--label-order", LABEL_ORDER], check=True)

## Theo dõi kết quả
Checkpoint và log lưu trong RUN_ROOT. Xem README để resume từ last.pth với cùng YAML/CLI; lịch LR và trạng thái RNG được giữ tại ranh giới epoch.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
run = Path(RUN_ROOT) / RUN_ID
metrics = pd.read_csv(run / "metrics.csv")
display(metrics.tail())
metrics.plot(x="epoch", y=["train/loss", "val/loss"], title="Loss")
metrics.plot(x="epoch", y=["val/mean_dice", "val/mean_iou"], title="Pixel-pooled validation metrics")
plt.show()